# Loudspeaker price prediction

Initial data loading, quick checks, and simple split between numerical and categorical features.


## Project context

The original data comes from a loudspeaker supplier database I found on Reddit about two years ago (~3,000 entries), but many fields were partial. Based on the Thiele–Small minimal parameter set, I filtered the dataset to keep only speakers with complete values for:

$$
R_e,\; Q_{es},\; Q_{ms},\; f_s,\; S_d,\; V_{as}
$$

These parameters describe the physical behavior of the driver, and most of the other low‑frequency properties can be derived from them (see `Lot0_minimal_ts_explanation.md`). I then selected a few additional columns that might correlate with price, such as efficiency, nominal/max power, and magnet material. The magnet type is expected to matter (ferrite vs. neodymium, for example), while other fields are exploratory. One of my curiosities is whether price can cluster by brand when the physical parameters are similar.


In [20]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## LOT 1.1 Data loading and first observations


In [21]:
data_path = '../Datas/speaker_db_selected_refined.csv'
df = pd.read_csv(data_path)
df.head()


,row_id,price,spec_general_marque,spec_general_type_produit,spec_general_reference,spec_informations_impedance_nominale,spec_forme_materiaux_systeme_magnetique,spec_forme_materiaux_forme_facade,spec_parametres_petits_signaux_qts,spec_parametres_petits_signaux_qes,spec_parametres_petits_signaux_qms,spec_forme_materiaux_materiau_saladier,spec_forme_materiaux_materiau_suspension,spec_forme_materiaux_support_bobine,spec_forme_materiaux_fil_bobine,spec_forme_materiaux_materiau_dome,spec_informations_puissance_nominale_w,spec_informations_sensibilite_fabricant_db,spec_parametres_petits_signaux_fs_hz,spec_donnees_poids_kg,spec_parametres_fondamentaux_re_ohm,spec_donnees_xmax_mm,spec_parametres_fondamentaux_mms_gr,spec_donnees_ebp_hz,spec_parametres_petits_signaux_vas_l,spec_donnees_rendement_calcule_pct,spec_parametres_fondamentaux_le_mh,spec_parametres_fondamentaux_sd_cm2,spec_parametres_fondamentaux_bl_t_m,spec_dimensions_diametre_systeme_magnetique_mm,spec_informations_puissance_max_w,spec_dimensions_hauteur_entrefer_mm,spec_dimensions_hauteur_bobinage_mm,spec_donnees_rendement_pct
0,1,63.0,Audax,Haut parleur a cone,HT170G8,8+8 ohm,Ferrite,Non cylindrique,0.49,0.55,4.93,Acier,NaN,Aluminium,NaN,NaN,50.0,90.0,45.0,1.3,3.50,3.50,17.40,82.0,19.9,0.32,0.49,136.0,5.50,85.8,NaN,5.0,12.0,NaN
1,2,294.0,Audax,Haut parleur bicone,AM21LB25ALBC,6 ohm,Ferrite,Non cylindrique,0.40,0.42,7.14,Zamak,Tissu,Nomex,CCAW (alu recouvert de cuivre),NaN,35.0,95.0,43.0,2.7,5.55,2.25,13.63,102.0,65.0,1.19,0.34,214.0,6.96,110.0,NaN,5.0,9.5,NaN
2,3,77.0,Audax,Haut parleur a cone,HM130Z0,8 ohm,Ferrite,Non cylindrique,0.31,0.32,12.16,Zamak,Caoutchouc,Kapton,Cuivre,NaN,50.0,92.0,68.0,1.2,6.40,2.00,5.90,213.0,8.3,0.79,0.22,80.0,7.10,85.8,NaN,5.0,9.0,NaN
3,4,93.0,Audax,Haut parleur a cone,HM130Z4,8+8 ohm,Ferrite,Non cylindrique,0.23,0.24,5.20,Zamak,NaN,Kapton,Cuivre,NaN,50.0,91.0,46.0,1.2,3.00,3.25,10.40,192.0,10.2,0.40,0.30,80.0,6.20,85.8,NaN,5.0,11.5,NaN
4,5,102.0,Audax,Haut parleur a cone,HM170Z0,8 ohm,Ferrite,Cylindrique,0.40,0.42,6.15,Zamak,Caoutchouc,Kapton,Cuivre,NaN,60.0,91.0,40.0,NaN,6.20,3.75,9.90,95.0,45.3,0.67,0.34,138.0,6.00,86.1,NaN,5.0,12.5,NaN


In [22]:
df.shape


(1530, 34)

In [23]:
df.dtypes


row_id                                              int64
price                                             float64
spec_general_marque                                object
spec_general_type_produit                          object
spec_general_reference                             object
spec_informations_impedance_nominale               object
spec_forme_materiaux_systeme_magnetique            object
spec_forme_materiaux_forme_facade                  object
spec_parametres_petits_signaux_qts                float64
spec_parametres_petits_signaux_qes                float64
spec_parametres_petits_signaux_qms                float64
spec_forme_materiaux_materiau_saladier             object
spec_forme_materiaux_materiau_suspension           object
spec_forme_materiaux_support_bobine                object
spec_forme_materiaux_fil_bobine                    object
spec_forme_materiaux_materiau_dome                 object
spec_informations_puissance_nominale_w            float64
spec_informati

## LOT 1.2 Missing values overview


In [24]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)).round(4)
pd.DataFrame({'missing': missing, 'missing_pct': missing_pct})


,missing,missing_pct
spec_forme_materiaux_materiau_dome,1273,0.8320
spec_donnees_rendement_pct,687,0.4490
spec_dimensions_diametre_systeme_magnetique_mm,595,0.3889
spec_informations_puissance_max_w,516,0.3373
spec_forme_materiaux_fil_bobine,482,0.3150
spec_dimensions_hauteur_bobinage_mm,473,0.3092
spec_forme_materiaux_support_bobine,443,0.2895
spec_forme_materiaux_materiau_suspension,384,0.2510
spec_dimensions_hauteur_entrefer_mm,320,0.2092
spec_donnees_poids_kg,297,0.1941


  With these columns having heavy missingness, it’s not necessarily a problem for tree‑based models (XGBoost can
  handle missing values), but it can hurt linear models unless you impute or drop them. A practical approach is
  to keep them for tree models and compare against a version where they’re removed.
  Some of them are interesting feature, so i'll try with and remove them if problematic.


## LOT 1.3 Numerical vs categorical split


In [25]:
numeric_cols = df.select_dtypes(include=['number']).columns
categorical_cols = df.select_dtypes(exclude=['number']).columns
len(numeric_cols), len(categorical_cols)


(23, 11)

In [26]:
numeric_cols


Index(['row_id', 'price', 'spec_parametres_petits_signaux_qts', 'spec_parametres_petits_signaux_qes', 'spec_parametres_petits_signaux_qms', 'spec_informations_puissance_nominale_w',
       'spec_informations_sensibilite_fabricant_db', 'spec_parametres_petits_signaux_fs_hz', 'spec_donnees_poids_kg', 'spec_parametres_fondamentaux_re_ohm', 'spec_donnees_xmax_mm',
       'spec_parametres_fondamentaux_mms_gr', 'spec_donnees_ebp_hz', 'spec_parametres_petits_signaux_vas_l', 'spec_donnees_rendement_calcule_pct', 'spec_parametres_fondamentaux_le_mh',
       'spec_parametres_fondamentaux_sd_cm2', 'spec_parametres_fondamentaux_bl_t_m', 'spec_dimensions_diametre_systeme_magnetique_mm', 'spec_informations_puissance_max_w',
       'spec_dimensions_hauteur_entrefer_mm', 'spec_dimensions_hauteur_bobinage_mm', 'spec_donnees_rendement_pct'],
      dtype='object')

In [27]:
categorical_cols


Index(['spec_general_marque', 'spec_general_type_produit', 'spec_general_reference', 'spec_informations_impedance_nominale', 'spec_forme_materiaux_systeme_magnetique',
       'spec_forme_materiaux_forme_facade', 'spec_forme_materiaux_materiau_saladier', 'spec_forme_materiaux_materiau_suspension', 'spec_forme_materiaux_support_bobine',
       'spec_forme_materiaux_fil_bobine', 'spec_forme_materiaux_materiau_dome'],
      dtype='object')

In [28]:
# Sort columns: numeric first, then non-numeric
numeric_cols = df.select_dtypes(include=['number']).columns
categorical_cols = df.select_dtypes(exclude=['number']).columns
ordered_cols = list(numeric_cols) + list(categorical_cols)

# Explore unique values, missing values, and data types for each column
for col in ordered_cols:
    dtype = df[col].dtype
    missing = df[col].isna().sum()
    unique = df[col].nunique()
    sample = df[col].dropna().unique()[:5]

    print(f"{col} ({dtype})")
    print(f"  Unique: {unique:,} | Missing: {missing:,}")
    print(f"  Sample: {list(sample)}")
    print()


row_id (int64)
  Unique: 1,530 | Missing: 0
  Sample: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

price (float64)
  Unique: 454 | Missing: 0
  Sample: [np.float64(63.0), np.float64(294.0), np.float64(77.0), np.float64(93.0), np.float64(102.0)]

spec_parametres_petits_signaux_qts (float64)
  Unique: 143 | Missing: 1
  Sample: [np.float64(0.49), np.float64(0.4), np.float64(0.31), np.float64(0.23), np.float64(0.39)]

spec_parametres_petits_signaux_qes (float64)
  Unique: 173 | Missing: 0
  Sample: [np.float64(0.55), np.float64(0.42), np.float64(0.32), np.float64(0.24), np.float64(0.73)]

spec_parametres_petits_signaux_qms (float64)
  Unique: 635 | Missing: 0
  Sample: [np.float64(4.93), np.float64(7.14), np.float64(12.16), np.float64(5.2), np.float64(6.15)]

spec_informations_puissance_nominale_w (float64)
  Unique: 68 | Missing: 2
  Sample: [np.float64(50.0), np.float64(35.0), np.float64(60.0), np.float64(70.0), np.float64(120.0)]

spec_informations_sensibilite_fab

### Column descriptions

**Numerical columns**
- **row_id**: Sequential row index (not a feature).
- **price**: Loudspeaker price in euros .
- **spec_parametres_petits_signaux_qts**: Total Q factor of the driver.
- **spec_parametres_petits_signaux_qes**: Electrical Q factor.
- **spec_parametres_petits_signaux_qms**: Mechanical Q factor.
- **spec_informations_puissance_nominale_w**: Nominal power handling in watts.
- **spec_parametres_petits_signaux_fs_hz**: Resonance frequency (Fs) in Hz.
- **spec_donnees_poids_kg**: Driver weight in kilograms.
- **spec_parametres_fondamentaux_re_ohm**: DC resistance Re in ohms.
- **spec_donnees_xmax_mm**: Linear excursion Xmax in mm.
- **spec_parametres_fondamentaux_mms_gr**: Moving mass Mms in grams.
- **spec_donnees_ebp_hz**: EBP (efficiency bandwidth product) in Hz.
- **spec_parametres_petits_signaux_vas_l**: Equivalent compliance volume Vas in liters.
- **spec_donnees_rendement_calcule_pct**: Calculated efficiency in percent.
- **spec_parametres_fondamentaux_le_mh**: Voice coil inductance Le in mH.
- **spec_parametres_fondamentaux_sd_cm2**: Cone area Sd in cm².
- **spec_parametres_fondamentaux_bl_t_m**: Force factor Bl in T·m.
- **spec_dimensions_diametre_systeme_magnetique_mm**: Magnet system diameter in mm.
- **spec_informations_puissance_max_w**: Maximum power in watts.
- **spec_dimensions_hauteur_entrefer_mm**: Gap height in mm.
- **spec_dimensions_hauteur_bobinage_mm**: Voice coil height in mm.
- **spec_donnees_rendement_pct**: Efficiency (if provided) in percent.

**Categorical columns**
- **spec_general_marque**: Brand.
- **spec_general_type_produit**: Product type (cone, bicone, coaxial, tweeter).
- **spec_general_reference**: Manufacturer reference/model identifier.
- **spec_informations_impedance_nominale**: Nominal impedance (categorical, e.g., `8 ohm`, `4+8 ohm`).
- **spec_forme_materiaux_systeme_magnetique**: Magnet type (ferrite, neodymium, etc.).
- **spec_forme_materiaux_forme_facade**: Frame/baffle shape.
- **spec_forme_materiaux_materiau_saladier**: Basket material.
- **spec_forme_materiaux_materiau_suspension**: Surround/suspension material.
- **spec_forme_materiaux_support_bobine**: Voice coil former material.
- **spec_forme_materiaux_fil_bobine**: Voice coil wire type.
- **spec_forme_materiaux_materiau_dome**: Dome/diaphragm material (if applicable).


## LOT 1.4 Numerical summary


In [32]:
df[numeric_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
row_id,1530.0,765.500000,441.817270,1.00,383.2500,765.500,1147.7500,1530.00
price,1530.0,201.902614,162.344659,4.00,84.2500,162.000,274.0000,933.00
spec_parametres_petits_signaux_qts,1529.0,0.453848,0.340868,0.12,0.2900,0.360,0.4800,4.30
spec_parametres_petits_signaux_qes,1530.0,0.546323,0.647936,0.12,0.3100,0.390,0.5400,10.02
spec_parametres_petits_signaux_qms,1530.0,6.318407,4.392370,1.09,3.7725,5.500,7.9875,111.81
spec_informations_puissance_nominale_w,1528.0,388.884162,431.004093,1.00,90.0000,250.000,500.0000,2500.00
spec_informations_sensibilite_fabricant_db,1517.0,93.936170,5.025874,39.00,90.5000,95.100,97.6000,104.00
spec_parametres_petits_signaux_fs_hz,1530.0,71.854830,70.175429,6.00,39.0000,52.000,77.4500,1021.70
spec_donnees_poids_kg,1233.0,5.508491,7.474734,0.10,1.6000,3.600,7.7000,92.00
spec_parametres_fondamentaux_re_ohm,1530.0,5.717340,2.490968,1.30,4.9000,5.400,6.1275,45.00


## LOT 1.5 Categorical summary


In [33]:
cat_summary = pd.DataFrame({
    'nunique': df[categorical_cols].nunique(),
    'missing': df[categorical_cols].isna().sum(),
}).sort_values('nunique', ascending=False)
cat_summary


,nunique,missing
spec_general_reference,1230,37
spec_general_marque,22,0
spec_informations_impedance_nominale,18,5
spec_forme_materiaux_support_bobine,11,443
spec_forme_materiaux_materiau_dome,11,1273
spec_forme_materiaux_materiau_suspension,11,384
spec_forme_materiaux_materiau_saladier,7,157
spec_general_type_produit,5,0
spec_forme_materiaux_systeme_magnetique,5,29
spec_forme_materiaux_forme_facade,5,109


In [34]:
# Show top values for low-cardinality categorical features
for col in categorical_cols:
    if df[col].nunique() <= 15:
        print(f'\n{col}')
        print(df[col].value_counts(dropna=False).head(10))



spec_general_type_produit
spec_general_type_produit
Haut parleur a cone                    1347
Haut-parleur coaxial a deux entrees     121
Haut parleur bicone                      39
Haut-parleur coaxial a une entree        17
Tweeter a dome                            6
Name: count, dtype: int64

spec_forme_materiaux_systeme_magnetique
spec_forme_materiaux_systeme_magnetique
Ferrite                907
Neodymium              567
NaN                     29
Ferrite + Neodymium     19
Alnico                   7
Excitation               1
Name: count, dtype: int64

spec_forme_materiaux_forme_facade
spec_forme_materiaux_forme_facade
Cylindrique        1082
Non cylindrique     313
NaN                 109
Carre                24
Elliptique            1
Rectangle             1
Name: count, dtype: int64

spec_forme_materiaux_materiau_saladier
spec_forme_materiaux_materiau_saladier
Aluminium        806
Acier            488
NaN              157
Zamak             39
Plastique         21
Magnesium

## Notes
- Cleaning and column selection were handled upstream.
- Next steps: decide final features, handle missing values, and set a baseline model.
